# Topic 1: Exploratory Data Analysis with Pandas

**Exploratory data analysis (EDA)**: the step before any modeling: you load the data, look at its shape and types, summarize distributions, and slice it in different ways to build intuition and check simple assumptions. Historically this was the whole data-analysis process before ML existed, and it still pays off — you often discover a simple "if-else" rule that a fancy model must beat to justify itself

## What Pandas is, and its two core objects

**Pandas** is a Python library for analyzing tabular data (`.csv`, `.tsv`, `.xlsx`). It lets you load, filter, aggregate, and reshape tables with SQL-like operations, and pairs naturally with Matplotlib/Seaborn for plots

Two data structures do all the work:

- **Series**: a one-dimensional labeled array holding values of a *single* type (like one column, plus an index). The **index** is the set of row labels attached to the values.
- **DataFrame**: a two-dimensional table; think of it as a dict of Series sharing one index. Each **column** is a Series of one type.

Vocabulary that recurs all course:
- An **instance** (a.k.a. observation, sample) is one **row** — here, one telecom customer
- A **feature** = one **column** is a measured attribute of the instance (e.g. `Total day minutes`)
- **Churn** = a customer leaving the service. It's the **target** we ultimately want to predict; `1` = left, `0` = loyal.

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)

## Loading data and first look

`read_csv` reads a CSV file (local path or URL) into a DataFrame. `head(n)` shows the first `n` rows (default 5) — the reflex first move to eyeball what you loaded.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"
df = pd.read_csv(DATA_URL + "telecom_churn.csv")
df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


The dataset has **3333 rows × 20 columns** — 3333 customers, 20 features (state, plan flags, call minutes/charges by time of day, customer-service calls, and the `Churn` target).

Three quick inspectors:
- `df.shape` → `(rows, columns)` tuple.
- `df.columns` → the column labels (an Index object).
- `df.info()` → per-column non-null counts and **dtype**, plus memory use.

A **dtype** is the storage type of a column's values: `int64`/`float64` (numbers), `bool` (True/False), `object` (usually strings). `info()` is also how you spot **missing values** — if a column's non-null count is below the row count, some cells are empty. Here every column has 3333 non-nulls, so nothing is missing.

In [4]:
print(df.shape)
df.info()

(3333, 20)
<class 'pandas.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   str    
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   str    
 4   Voice mail plan         3333 non-null   str    
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   

### Changing a column's type: `astype`

`astype` casts a column to another dtype. Converting the boolean `Churn` to `int64` (True→1, False→0) makes it easy to average and cross-tabulate later — the mean of a 0/1 column is directly the fraction of 1s.

In [5]:
df["Churn"] = df["Churn"].astype("int64")

## Summary statistics

`describe()` reports, for every **numeric** column, the count, mean, standard deviation, min, max, median, and the 25%/75% quartiles. A **quartile** is a cut point: the 25% quartile is the value below which a quarter of the data falls; the median (50%) is the middle value. This is your first read on scale and spread of each feature.

In [6]:
df.describe()

,Account length,Area code,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
count,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00
mean,101.06,437.18,8.10,179.78,100.44,30.56,200.98,100.11,17.08,200.87,100.11,9.04,10.24,4.48,2.76,1.56,0.14
std,39.82,42.37,13.69,54.47,20.07,9.26,50.71,19.92,4.31,50.57,19.57,2.28,2.79,2.46,0.75,1.32,0.35
min,1.00,408.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,23.20,33.00,1.04,0.00,0.00,0.00,0.00,0.00
25%,74.00,408.00,0.00,143.70,87.00,24.43,166.60,87.00,14.16,167.00,87.00,7.52,8.50,3.00,2.30,1.00,0.00
50%,101.00,415.00,0.00,179.40,101.00,30.50,201.40,100.00,17.12,201.20,100.00,9.05,10.30,4.00,2.78,1.00,0.00
75%,127.00,510.00,20.00,216.40,114.00,36.79,235.30,114.00,20.00,235.30,113.00,10.59,12.10,6.00,3.27,2.00,0.00
max,243.00,510.00,51.00,350.80,165.00,59.64,363.70,170.00,30.91,395.00,175.00,17.77,20.00,20.00,5.40,9.00,1.00


To summarize **non-numeric** columns instead, pass `include=`. For `object`/`bool` columns you get `count`, `unique` (number of distinct values), `top` (most frequent), and `freq` (how often the top appears). E.g. there are 51 distinct states; `International plan` is "No" for 3010 of 3333 customers.

In [7]:
df.describe(include=["object", "bool"])

/var/folders/qq/4_f4lrts1qv6wzvprrqmy6p80000gn/T/ipykernel_56421/3870744420.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=["object", "bool"])


,State,International plan,Voice mail plan
count,3333,3333,3333
unique,51,2,2
top,WV,No,No
freq,106,3010,2411


### Counting categories: `value_counts`

`value_counts()` tallies how many times each distinct value appears in a Series — the go-to for categorical/boolean columns. Add `normalize=True` to get **fractions** instead of raw counts.

For `Churn`: 2850 loyal (0) vs 483 churned (1) — about **14.5% churn**. That's a bad rate for a real company, but the imbalance also matters for modeling: a model that always guesses "loyal" is already right ~85.5% of the time (more on that at the end).

In [8]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True)

Churn
0    0.86
1    0.14
Name: proportion, dtype: float64

## Sorting

`sort_values(by=...)` returns the DataFrame reordered by one or more columns. `ascending=False` sorts high-to-low; pass a *list* of booleans to set the direction per column when sorting by several. Sorting doesn't change the data, only its order — handy for spotting extremes.

In [9]:
df.sort_values(by="Total day charge", ascending=False).head()

# sort by several columns, each with its own direction
df.sort_values(by=["Churn", "Total day charge"], ascending=[True, False]).head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
688,MN,13,510,No,Yes,21,315.6,105,53.65,208.9,71,17.76,260.1,123,11.70,12.1,3,3.27,3,0
2259,NC,210,415,No,Yes,31,313.8,87,53.35,147.7,103,12.55,192.7,97,8.67,10.1,7,2.73,3,0
534,LA,67,510,No,No,0,310.4,97,52.77,66.5,123,5.65,246.5,99,11.09,9.2,10,2.48,4,0
575,SD,114,415,No,Yes,36,309.9,90,52.68,200.3,89,17.03,183.5,105,8.26,14.2,2,3.83,1,0
2858,AL,141,510,No,Yes,28,308.0,123,52.36,247.8,128,21.06,152.9,103,6.88,7.4,3,2.00,1,0


## Indexing and retrieving data

Several ways to pull out pieces of a DataFrame:

**Single column** — `df["Churn"]` returns that Series. Because `Churn` is 0/1, `df["Churn"].mean()` gives the churn proportion directly (0.145).

In [10]:
df["Churn"].mean()

np.float64(0.14491449144914492)

### Boolean indexing (filtering rows)

**Boolean indexing** keeps only the rows where a condition is True. The condition `df["Churn"] == 1` produces a boolean Series (True/False per row); `df[that]` returns just the matching rows. It's the workhorse for "show me the subset where…" questions.

Combine conditions with `&` (and) / `|` (or), and **wrap each condition in parentheses** — operator precedence bites otherwise.

In [11]:
# Average day-minutes among churned customers
df[df["Churn"] == 1]["Total day minutes"].mean()   # ~206.9

# Max international minutes among loyal customers with no international plan
df[(df["Churn"] == 0) & (df["International plan"] == "No")]["Total intl minutes"].max()  # 18.9

# Average of all numeric features for churned users
df.select_dtypes(include=np.number)[df["Churn"] == 1].mean()

Account length            102.66
Area code                 437.82
Number vmail messages       5.12
Total day minutes         206.91
Total day calls           101.34
Total day charge           35.18
Total eve minutes         212.41
Total eve calls           100.56
Total eve charge           18.05
Total night minutes       205.23
Total night calls         100.40
Total night charge          9.24
Total intl minutes         10.70
Total intl calls            4.16
Total intl charge           2.89
Customer service calls      2.23
Churn                       1.00
dtype: float64

### Label vs. position: `.loc` and `.iloc`

Two explicit indexers, easy to confuse:

- **`.loc[rows, cols]`** selects by **label** (names). Label slices are **inclusive** of both ends: `df.loc[0:5, "State":"Area code"]` returns rows 0 through 5 *and* every column from `State` to `Area code`.
- **`.iloc[rows, cols]`** selects by **integer position**. Position slices follow normal Python rules — the **end is excluded**: `df.iloc[0:5, 0:3]` is the first 5 rows and first 3 columns.

(`df[:1]` / `df[-1:]` still work for grabbing the first / last row.)

In [12]:
df.loc[0:5, "State":"Area code"]   # by label, endpoints included
df.iloc[0:5, 0:3]                  # by position, end excluded

,State,Account length,Area code
0,KS,128,415
1,OH,107,415
2,NJ,137,415
3,OH,84,408
4,OK,75,415


## Applying functions to cells, columns, and rows

- **`apply(func)`** runs `func` on each **column** by default (e.g. `df.apply(np.max)` → max of every column). Pass **`axis=1`** to run it across each **row** instead. Lambdas are handy here.
- **`map(dict)`** on a Series replaces values via a `{old: new}` mapping — great for recoding categories.

Note the subtle difference between `map` and `replace`: for a value **not** in the mapping dictionary, `map` sets it to **NaN** (missing), whereas `replace` leaves it untouched. Prefer `replace` when you only want to remap *some* values.

In [13]:
df.apply(np.max)                                   # max of each column

# axis=1 applies per row; here filter states starting with 'W'
df[df["State"].apply(lambda state: state[0] == "W")].head()

# recode Yes/No -> True/False
d = {"No": False, "Yes": True}
df["International plan"] = df["International plan"].map(d)
df = df.replace({"Voice mail plan": d})   # replace leaves unmapped values as-is

## Grouping: split–apply–combine

**`groupby`** implements the split–apply–combine pattern, the single most useful aggregation tool:

```
df.groupby(by=grouping_columns)[columns_to_show].function()
```

1. **Split** — partition rows into groups by the distinct values of `grouping_columns` (those values become the new index).
2. **Select** — pick the columns to summarize (`columns_to_show`).
3. **Apply** — compute a statistic per group.

Use **`.agg([...])`** to apply *several* functions at once. Below, grouping by `Churn` shows that churned customers average noticeably more daytime minutes than loyal ones — an early signal that call-usage relates to churn.

In [14]:
columns_to_show = ["Total day minutes", "Total eve minutes", "Total night minutes"]

df.groupby(["Churn"])[columns_to_show].describe(percentiles=[])

# equivalent, choosing the statistics explicitly
df.groupby(["Churn"])[columns_to_show].agg(["mean", "std", "min", "max"])

Total day minutes                    Total eve minutes               \
                   mean    std  min    max              mean    std   min   
Churn                                                                       
0                175.18  50.18  0.0  315.6            199.04  50.29   0.0   
1                206.91  69.00  0.0  350.8            212.41  51.73  70.9   

             Total night minutes                      
         max                mean    std   min    max  
Churn                                                 
0      361.8              200.13  51.11  23.2  395.0  
1      363.7              205.23  47.13  47.4  354.9

## Summary tables: crosstab and pivot_table

A **contingency table** (a.k.a. cross-tabulation) counts how many rows fall into each combination of two categorical variables — it's how you see whether two categories move together.

- **`pd.crosstab(a, b)`** builds that count table. `normalize=True` gives proportions instead of counts; `margins=True` adds row/column totals (an `All` line).
- **`pivot_table`** is the spreadsheet-style generalization. Key arguments:
  - `values` — column(s) to compute statistics on,
  - `index` — column(s) to group rows by,
  - `aggfunc` — the statistic (`"mean"`, `"sum"`, `"max"`, …).

In [15]:
pd.crosstab(df["Churn"], df["International plan"])
pd.crosstab(df["Churn"], df["Voice mail plan"], normalize=True)

df.pivot_table(
    ["Total day calls", "Total eve calls", "Total night calls"],
    ["Area code"],
    aggfunc="mean",
)

,Total day calls,Total eve calls,Total night calls
Area code,,,
408,100.50,99.79,99.04
415,100.58,100.50,100.40
510,100.10,99.67,100.60


## DataFrame transformations: add and drop

**Add a column** — assign a computed Series to a new name: `df["Total charge"] = df["Total day charge"] + ...`. For explicit placement use `df.insert(loc, column, value)` (`loc` = position to insert at).

**Drop rows/columns** — `df.drop(labels, axis=...)`. `axis=1` drops **columns**, `axis=0` (default) drops **rows**. `inplace=True` mutates the DataFrame in place; `inplace=False` (default) returns a modified copy and leaves the original alone.

In [16]:
# add a column directly from a computed expression
df["Total charge"] = (
    df["Total day charge"] + df["Total eve charge"]
    + df["Total night charge"] + df["Total intl charge"]
)

# drop columns (axis=1, in place); dropping rows uses axis=0
df.drop(["Total charge"], axis=1, inplace=True)
df.drop([1, 2]).head()   # returns a copy without rows 1 and 2

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,False,True,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,0
3,OH,84,408,True,False,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,0
4,OK,75,415,True,False,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,0
5,AL,118,510,True,False,0,223.4,98,37.98,220.6,101,18.75,203.9,118,9.18,6.3,6,1.70,0,0
6,MA,121,510,False,True,24,218.2,88,37.09,348.5,108,29.62,212.6,118,9.57,7.5,7,2.03,3,0


## First attempt at "predicting" churn — without any ML

EDA alone can produce a decent predictor. The idea: find features that clearly separate churners from loyal customers, then write a simple rule.

**Signal 1 — International plan.** The crosstab (with `margins=True`) shows churn is far more common among customers who have the international plan. Plausibly, high/poorly-managed international charges breed discontent.

In [17]:
pd.crosstab(df["Churn"], df["International plan"], margins=True)

International plan,False,True,All
Churn,,,
0,2664,186,2850
1,346,137,483
All,3010,323,3333


**Signal 2 — Customer service calls.** Cross-tabulating churn against the number of support calls shows the churn rate spikes sharply once a customer makes **4 or more** calls — a classic "this person is frustrated" indicator. A `countplot` (Seaborn) makes it obvious visually; visual EDA is Topic 2.

In [18]:
pd.crosstab(df["Churn"], df["Customer service calls"], margins=True)

# Turn the threshold into a binary feature: more than 3 service calls
df["Many_service_calls"] = (df["Customer service calls"] > 3).astype("int")
pd.crosstab(df["Many_service_calls"], df["Churn"], margins=True)

Churn,0,1,All
Many_service_calls,,,
0,2721,345,3066
1,129,138,267
All,2850,483,3333


**Combine the two signals** into one rule and check it against the data:

```
Churn = 1  if  (International plan == True)  AND  (Customer service calls > 3)
Churn = 0  otherwise
```

In [19]:
pd.crosstab(df["Many_service_calls"] & df["International plan"], df["Churn"], margins=True)

Churn,0,1,All
row_0,,,
False,2841,464,3305
True,9,19,28
All,2850,483,3333


### Baselines: the numbers your ML must beat

A **baseline** is the accuracy of a trivially simple predictor — the bar any "real" model has to clear to be worth its complexity. **Accuracy** = fraction of predictions that are correct.

Two baselines from pure EDA:

- **85.5% — the naive constant model.** 85.5% of customers are loyal, so *always* predicting "loyal" is right 85.5% of the time. Any model scoring below this is worthless.
- **85.8% — the two-condition rule** (`International plan AND >3 service calls → churn`). From the last crosstab it's wrong only 464 + 9 times, so accuracy ≈ 1 − (464+9)/3333 ≈ **85.8%** — a hair above naive, from hand-written logic.

Why this matters:
- These baselines are the **starting line** for every later model. Decision trees (Topic 3) will *learn* rules like this one **automatically** from the data.
- If a complex model beats 85.8% by only ~0.5%, that's a red flag — maybe the effort isn't justified and a two-line if-else suffices.
- **Always** wrangle the data, plot it, and check simple assumptions *before* reaching for heavy models. In industry you ship the simple solution first, then iterate.

## Takeaways

- **EDA first.** Look at shape, dtypes, missing values, and distributions before modeling — it builds intuition and catches problems early.
- **Series = one column; DataFrame = the table.** Rows are instances, columns are features.
- **Inspect:** `head`, `shape`, `columns`, `info`, `describe`, `value_counts`.
- **Filter** with boolean indexing (parenthesize each condition, combine with `&`/`|`).
- **Select** with `.loc` (labels, end inclusive) vs `.iloc` (positions, end exclusive).
- **Transform** with `apply`/`map`/`replace`; **aggregate** with `groupby(...).agg(...)`.
- **Summarize relationships** with `crosstab` and `pivot_table`.
- **Establish baselines** (naive constant + simple rule) so you know what any ML model must beat.

## Summary

This module walks through Pandas end-to-end on a telecom-churn dataset: loading and inspecting a DataFrame, computing summary statistics, filtering with boolean indexing, selecting by label vs. position, transforming columns, and aggregating with `groupby`, `crosstab`, and `pivot_table`. It closes by using EDA alone to build two baselines — a 85.5% "always loyal" constant and an 85.8% two-condition rule — that set the bar for the machine-learning models in later topics. The lesson: understand and slice your data with simple tools first; complex models must justify themselves against these cheap baselines.

## Glossary / Terms

- **EDA** = exploring/summarizing data before modeling
- **churn** = a customer leaving (target: 1=left, 0=loyal)
- **Series** = 1-D labeled single-type array
- **DataFrame** = 2-D table of Series
    - **instance** = a row/observation
    - **feature** = a column/attribute
    - **index** = row labels
    - **dtype** = a column's storage type (int64/float64/bool/object)
- **boolean indexing** = keep rows where a condition is True
- **`.loc`** = select by label (end inclusive)
- **`.iloc`** = select by integer position (end exclusive)
- **`groupby`** = split–apply–combine aggregation
    - **`agg`** = apply several functions per group
- **crosstab** = contingency table counting category combinations
- **pivot_table** = spreadsheet-style grouped aggregation
- **baseline** = accuracy of a trivial predictor
- **accuracy** = fraction of correct predictions